# ksnctf Problem 11 - Riddle

## 問題概要
複数の謎かけに答え、最終的にFLAGを入力する問題。
4問目「What is the flag?」の答えをリバースエンジニアリングで求める。

## 使用ツール
- x32dbg（動的解析）
- Rust（復号スクリプト）

## 解析手順

### 1. 暗号化方式の特定
x32dbgで`sub_6B17D0`を解析した結果、**RC4暗号**が使われていることが判明。
- Sボックス: `0x6B5578`（256バイト）
- XOR命令: `xor byte ptr ds:[eax+esi], cl`

### 2. 既知平文攻撃
problem1 Test Problemの答え`FLAG_SRORGLnTh2Q5fTwu`を仮入力し、以下の値をx32dbgで取得：
- 暗号化済み入力（ESI）
- 正解暗号文（ECX）

### 3. キーストリーム逆算
XORの対称性を利用：
```
キーストリーム = 既知平文 XOR 暗号化済み入力
正解          = 正解暗号文 XOR キーストリーム
```

In [4]:
use std::fs;
{
    // 入力文字列（既知平文）
    let plaintext_input: &[u8] = b"FLAG_SRORGLnTh2Q5fTwu";

    // 暗号化済み入力（ESIが指すメモリの内容）
    let encrypted_input: &[u8] = &[
        0xBF, 0xFF, 0x1B, 0x0D, 0x47, 0xB9, 0x0F, 0x79,
        0xCB, 0xA5, 0x4A, 0x51, 0xF5, 0x48, 0x75, 0x4E,
        0xC0, 0xE0, 0xF7, 0xDB, 0xB2,
    ];

    // 正解暗号文（ECXが指すメモリの内容）
    let correct_encrypted: &[u8] = &[
        0xBF, 0xFF, 0x1B, 0x0D, 0x47, 0xA7, 0x18, 0x4F,
        0xCB, 0xD6, 0x5C, 0x59, 0x95, 0x61, 0x73, 0x57,
        0xB3, 0xD1, 0x94, 0x9F, 0xAC,
    ];

    // Step 1: キーストリームを逆算
    // キーストリーム = 既知平文 XOR 暗号化済み入力
    let keystream: Vec<u8> = plaintext_input
        .iter()
        .zip(encrypted_input.iter())
        .map(|(p, e)| p ^ e)
        .collect();

    println!("キーストリーム:");
    for (i, k) in keystream.iter().enumerate() {
        print!("{:02X} ", k);
        if (i + 1) % 8 == 0 { println!(); }
    }
    println!();

    // Step 2: 正解文字列を復元
    // 正解 = 正解暗号文 XOR キーストリーム
    let answer: Vec<u8> = correct_encrypted
        .iter()
        .zip(keystream.iter())
        .map(|(c, k)| c ^ k)
        .collect();

    fs::write("flag.txt", &answer).unwrap();
}

キーストリーム:
F9 B3 5A 4A 18 EA 5D 36 
99 E2 06 3F A1 20 47 1F 
F5 86 A3 AC C7 


()

## 考察

### 暗号方式
RC4は**ストリーム暗号**であり、同じキーとSボックスの状態であれば
暗号化と復号が同一の処理になる。

### 既知平文攻撃の有効性
FLAGの形式（`FLAG_`プレフィックス）が既知であるため、
RC4の内部状態を完全に解析しなくても正解を導出できた。

### 教訓
- RC4単体では既知平文に弱い
- ストリーム暗号のキーストリームは再利用してはならない

## 補足 x32dbg
### Riddle.exeをアタッチしてトレース
1. Riddle.exeを起動し、3問目まで頑張って解く
1. [ファイル]-[アタッチ]でRiddleを選択する
1. [CPU]タブ内で右クリックし、[検索]-[すべてのユーザーモジュール]-[文字列参照]を選択
1. "Correct！"をダブルクリックし、表示されたアセンブリを遡り、確認
1. アドレス006B19E5でサブルーチンをコールしている（call <riddle.sub_6B17D0>）
1. アドレス006B1A02で暗号化済み入力（ESI)と 正解暗号文（ECX）を比較している（cmp edx,dword ptr ds:[ecx]）
1. 次のステップで比較結果が一致しなければ、"Wrong!"の表示へジャンプ（jne riddle.6B1A7D）

アドレス006B19E5のcall <riddle.sub_6B17D0>の行をダブルクリックし、サブルーチンのアセンブリを確認
- アドレス006B1963のxor byte ptr ds:[eax+esi],clで、XORしてキーストリームを算出
- アドレス0x6B5578 はRC4のSボックス（256バイトの配列）
- xor byte ptr ds:[eax+esi], cl の cl が固定値ではなく毎回変化するキーストリーム。

<!DOCTYPE html>
<html lang="ja">
<head>
<meta charset="UTF-8">
<style>
  body { font-family: monospace; padding: 20px; }
  h2 { font-family: sans-serif; }
  table { border-collapse: collapse; margin-bottom: 40px; font-size: 13px; }
  th { background: #333; color: #fff; padding: 6px 10px; text-align: left; }
  td { padding: 5px 10px; border: 1px solid #ccc; white-space: nowrap; }
  tr:nth-child(even) { background: #f5f5f5; }
  tr.highlight { background: #ffcccc !important; font-weight: bold; }
</style>
</head>
<body>

<h2>表1: Sボックス（006B5578）</h2>
<table>
  <thead>
    <tr><th>アドレス</th><th>バイト列</th><th>ASCII</th></tr>
  </thead>
  <tbody>
    <tr><td>006B5578</td><td>38 49 1B 3B E6 63 DC 66 12 28 08 42 7C C2 BB 24</td><td>8I.;æcÜf.(.B|Â»$</td></tr>
    <tr><td>006B5588</td><td>C5 1C 89 2E 20 D9 5E B1 78 65 D6 55 27 1D 67 52</td><td>Å... Ù^±xeÖU'.gR</td></tr>
    <tr><td>006B5598</td><td>7B 33 76 04 06 A2 5A 83 82 0F 00 62 D3 30 AF 2F</td><td>{3v..¢Z....bÓ0¯/</td></tr>
    <tr><td>006B55A8</td><td>A3 CD A1 9B AE 6D EE 69 E8 91 F7 E0 D4 2D EF 03</td><td>£Í¡.®mîiè.÷àÔ-ï.</td></tr>
    <tr><td>006B55B8</td><td>ED 81 2B 21 56 85 45 FF 8E 84 BC 8B 7A 98 37 10</td><td>í.+!V.Eÿ..¼.z.7.</td></tr>
    <tr><td>006B55C8</td><td>FD 68 01 11 F4 C8 22 BD 75 FB B7 23 DA 15 14 A0</td><td>ýh..ôÈ"½uû·#Ú.. </td></tr>
    <tr><td>006B55D8</td><td>E4 32 A5 0B C9 26 F9 AC D1 97 BE DE 8F 51 02 29</td><td>ä2¥.É&amp;ù¬Ñ.¾Þ.Q.)</td></tr>
    <tr><td>006B55E8</td><td>79 0C 71 CF 5F 80 86 64 4C 13 F1 F0 90 1F 40 57</td><td>y.qÏ_..dL.ñð..@W</td></tr>
    <tr><td>006B55F8</td><td>D2 77 61 F8 D7 C7 B3 7E 6B 31 EC 1A 09 72 47 B4</td><td>Òwaø×Ç³~k1ì..rG´</td></tr>
    <tr><td>006B5608</td><td>CA 95 C1 99 A7 5B 94 C0 6C 9C 54 B8 D8 5D 1E E7</td><td>Ê.Á.§[.Àl.T¸Ø].ç</td></tr>
    <tr><td>006B5618</td><td>3C 60 7D BA 39 AD 07 34 93 DB 9F CE 88 B6 3A B9</td><td>&lt;`}º9..4.Û.Î.¶:¹</td></tr>
    <tr><td>006B5628</td><td>6E 58 59 B2 CC 0E 35 F5 D5 E9 CB DF BF 46 E1 8D</td><td>nXY²Ì.5õÕéËß¿Fá.</td></tr>
    <tr><td>006B5638</td><td>A6 4A 0A EB DD B0 AA 74 9E 7F 4F A4 E3 FA 6F 9A</td><td>¦J.ëÝ°ªt..O¤ãúo.</td></tr>
    <tr><td>006B5648</td><td>3F 3D E2 F3 C6 C4 6A 73 8C 48 9D E5 B5 4B 4E 18</td><td>?=âóÆÄjs.H.åµKN.</td></tr>
    <tr><td>006B5658</td><td>53 3E 87 92 05 5C F6 44 0D EA A8 43 A9 8A FE D0</td><td>S&gt;...\öD.ê¨C©.þÐ</td></tr>
    <tr><td>006B5668</td><td>41 50 AB FC F2 19 36 70 25 2C 4D 16 C3 17 2A 96</td><td>AP«üò.6p%,M.Ã.*.</td></tr>
  </tbody>
</table>

<h2>表2: sub_6B17D0 逆アセンブル</h2>
<table>
  <thead>
    <tr><th>アドレス</th><th>バイト列</th><th>ニーモニック</th><th>コメント</th></tr>
  </thead>
  <tbody>
    <tr><td>006B17D0</td><td>55</td><td>push ebp</td><td></td></tr>
    <tr><td>006B17D1</td><td>8BEC</td><td>mov ebp,esp</td><td></td></tr>
    <tr><td>006B17D3</td><td>83EC 0C</td><td>sub esp,C</td><td></td></tr>
    <tr><td>006B17D6</td><td>33C0</td><td>xor eax,eax</td><td>eax:"FLAG_SRORGLnTh2Q5fTwu"</td></tr>
    <tr><td>006B17D8</td><td>EB 06</td><td>jmp riddle.6B17E0</td><td></td></tr>
    <tr><td>006B17DA</td><td>8D9B 00000000</td><td>lea ebx,dword ptr ds:[ebx]</td><td>ebx:"\8k"</td></tr>
    <tr><td>006B17E0</td><td>8880 78556B00</td><td>mov byte ptr ds:[eax+6B5578],al</td><td></td></tr>
    <tr><td>006B17E6</td><td>40</td><td>inc eax</td><td>eax:"FLAG_SRORGLnTh2Q5fTwu"</td></tr>
    <tr><td>006B17E7</td><td>3D 00010000</td><td>cmp eax,100</td><td>eax:"FLAG_SRORGLnTh2Q5fTwu"</td></tr>
    <tr><td>006B17EC</td><td>7C F2</td><td>jl riddle.6B17E0</td><td></td></tr>
    <tr><td>006B17EE</td><td>53</td><td>push ebx</td><td>ebx:"\8k"</td></tr>
    <tr><td>006B17EF</td><td>BA 02000000</td><td>mov edx,2</td><td></td></tr>
    <tr><td>006B17F4</td><td>81EA 78556B00</td><td>sub edx,riddle.6B5578</td><td></td></tr>
    <tr><td>006B17FA</td><td>56</td><td>push esi</td><td></td></tr>
    <tr><td>006B17FB</td><td>57</td><td>push edi</td><td></td></tr>
    <tr><td>006B17FC</td><td>8955 F8</td><td>mov dword ptr ss:[ebp-8],edx</td><td></td></tr>
    <tr><td>006B17FF</td><td>BF 01000000</td><td>mov edi,1</td><td></td></tr>
    <tr><td>006B1804</td><td>BA 03000000</td><td>mov edx,3</td><td></td></tr>
    <tr><td>006B1809</td><td>32C9</td><td>xor cl,cl</td><td></td></tr>
    <tr><td>006B180B</td><td>33C0</td><td>xor eax,eax</td><td>eax:"FLAG_SRORGLnTh2Q5fTwu"</td></tr>
    <tr><td>006B180D</td><td>81EF 78556B00</td><td>sub edi,riddle.6B5578</td><td></td></tr>
    <tr><td>006B1813</td><td>81EA 78556B00</td><td>sub edx,riddle.6B5578</td><td></td></tr>
    <tr><td>006B1819</td><td>8955 F4</td><td>mov dword ptr ss:[ebp-C],edx</td><td></td></tr>
    <tr><td>006B181C</td><td>8D6424 00</td><td>lea esp,dword ptr ss:[esp]</td><td>[esp]:"FLAG_SRORGLnTh2Q5fTwu"</td></tr>
    <tr><td>006B1820</td><td>8A90 78556B00</td><td>mov dl,byte ptr ds:[eax+6B5578]</td><td></td></tr>
    <tr><td>006B1826</td><td>8BF0</td><td>mov esi,eax</td><td>eax:"FLAG_SRORGLnTh2Q5fTwu"</td></tr>
    <tr><td>006B1828</td><td>81E6 0F000080</td><td>and esi,8000000F</td><td></td></tr>
    <tr><td>006B182E</td><td>79 05</td><td>jns riddle.6B1835</td><td></td></tr>
    <tr><td>006B1830</td><td>4E</td><td>dec esi</td><td></td></tr>
    <tr><td>006B1831</td><td>83CE F0</td><td>or esi,FFFFFFF0</td><td></td></tr>
    <tr><td>006B1834</td><td>46</td><td>inc esi</td><td></td></tr>
    <tr><td>006B1835</td><td>0FB69E 18386B00</td><td>movzx ebx,byte ptr ds:[esi+6B3818]</td><td>ebx:"\8k"</td></tr>
    <tr><td>006B183C</td><td>02DA</td><td>add bl,dl</td><td></td></tr>
    <tr><td>006B183E</td><td>02CB</td><td>add cl,bl</td><td></td></tr>
    <tr><td>006B1840</td><td>0FB6F1</td><td>movzx esi,cl</td><td></td></tr>
    <tr><td>006B1843</td><td>8A9E 78556B00</td><td>mov bl,byte ptr ds:[esi+6B5578]</td><td></td></tr>
    <tr><td>006B1849</td><td>8896 78556B00</td><td>mov byte ptr ds:[esi+6B5578],dl</td><td></td></tr>
    <tr><td>006B184F</td><td>8A90 79556B00</td><td>mov dl,byte ptr ds:[eax+6B5579]</td><td></td></tr>
    <tr><td>006B1855</td><td>8DB407 78556B00</td><td>lea esi,dword ptr ds:[edi+eax+6B5578]</td><td></td></tr>
    <tr><td>006B185C</td><td>81E6 0F000080</td><td>and esi,8000000F</td><td></td></tr>
    <tr><td>006B1862</td><td>8898 78556B00</td><td>mov byte ptr ds:[eax+6B5578],bl</td><td></td></tr>
    <tr><td>006B1868</td><td>79 05</td><td>jns riddle.6B186F</td><td></td></tr>
    <tr><td>006B186A</td><td>4E</td><td>dec esi</td><td></td></tr>
    <tr><td>006B186B</td><td>83CE F0</td><td>or esi,FFFFFFF0</td><td></td></tr>
    <tr><td>006B186E</td><td>46</td><td>inc esi</td><td></td></tr>
    <tr><td>006B186F</td><td>0FB69E 18386B00</td><td>movzx ebx,byte ptr ds:[esi+6B3818]</td><td>ebx:"\8k"</td></tr>
    <tr><td>006B1876</td><td>02DA</td><td>add bl,dl</td><td></td></tr>
    <tr><td>006B1878</td><td>02CB</td><td>add cl,bl</td><td></td></tr>
    <tr><td>006B187A</td><td>0FB6F1</td><td>movzx esi,cl</td><td></td></tr>
    <tr><td>006B187D</td><td>8A9E 78556B00</td><td>mov bl,byte ptr ds:[esi+6B5578]</td><td></td></tr>
    <tr><td>006B1883</td><td>8896 78556B00</td><td>mov byte ptr ds:[esi+6B5578],dl</td><td></td></tr>
    <tr><td>006B1889</td><td>8B75 F8</td><td>mov esi,dword ptr ss:[ebp-8]</td><td></td></tr>
    <tr><td>006B188C</td><td>8A90 7A556B00</td><td>mov dl,byte ptr ds:[eax+6B557A]</td><td></td></tr>
    <tr><td>006B1892</td><td>8DB406 78556B00</td><td>lea esi,dword ptr ds:[esi+eax+6B5578]</td><td></td></tr>
    <tr><td>006B1899</td><td>81E6 0F000080</td><td>and esi,8000000F</td><td></td></tr>
    <tr><td>006B189F</td><td>8898 79556B00</td><td>mov byte ptr ds:[eax+6B5579],bl</td><td></td></tr>
    <tr><td>006B18A5</td><td>79 05</td><td>jns riddle.6B18AC</td><td></td></tr>
    <tr><td>006B18A7</td><td>4E</td><td>dec esi</td><td></td></tr>
    <tr><td>006B18A8</td><td>83CE F0</td><td>or esi,FFFFFFF0</td><td></td></tr>
    <tr><td>006B18AB</td><td>46</td><td>inc esi</td><td></td></tr>
    <tr><td>006B18AC</td><td>0FB69E 18386B00</td><td>movzx ebx,byte ptr ds:[esi+6B3818]</td><td>ebx:"\8k"</td></tr>
    <tr><td>006B18B3</td><td>02DA</td><td>add bl,dl</td><td></td></tr>
    <tr><td>006B18B5</td><td>02CB</td><td>add cl,bl</td><td></td></tr>
    <tr><td>006B18B7</td><td>0FB6F1</td><td>movzx esi,cl</td><td></td></tr>
    <tr><td>006B18BA</td><td>8A9E 78556B00</td><td>mov bl,byte ptr ds:[esi+6B5578]</td><td></td></tr>
    <tr><td>006B18C0</td><td>8896 78556B00</td><td>mov byte ptr ds:[esi+6B5578],dl</td><td></td></tr>
    <tr><td>006B18C6</td><td>8B75 F4</td><td>mov esi,dword ptr ss:[ebp-C]</td><td></td></tr>
    <tr><td>006B18C9</td><td>8A90 7B556B00</td><td>mov dl,byte ptr ds:[eax+6B557B]</td><td></td></tr>
    <tr><td>006B18CF</td><td>8DB406 78556B00</td><td>lea esi,dword ptr ds:[esi+eax+6B5578]</td><td></td></tr>
    <tr><td>006B18D6</td><td>81E6 0F000080</td><td>and esi,8000000F</td><td></td></tr>
    <tr><td>006B18DC</td><td>8898 7A556B00</td><td>mov byte ptr ds:[eax+6B557A],bl</td><td></td></tr>
    <tr><td>006B18E2</td><td>79 05</td><td>jns riddle.6B18E9</td><td></td></tr>
    <tr><td>006B18E4</td><td>4E</td><td>dec esi</td><td></td></tr>
    <tr><td>006B18E5</td><td>83CE F0</td><td>or esi,FFFFFFF0</td><td></td></tr>
    <tr><td>006B18E8</td><td>46</td><td>inc esi</td><td></td></tr>
    <tr><td>006B18E9</td><td>0FB69E 18386B00</td><td>movzx ebx,byte ptr ds:[esi+6B3818]</td><td>ebx:"\8k"</td></tr>
    <tr><td>006B18F0</td><td>02DA</td><td>add bl,dl</td><td></td></tr>
    <tr><td>006B18F2</td><td>02CB</td><td>add cl,bl</td><td></td></tr>
    <tr><td>006B18F4</td><td>0FB6F1</td><td>movzx esi,cl</td><td></td></tr>
    <tr><td>006B18F7</td><td>8A9E 78556B00</td><td>mov bl,byte ptr ds:[esi+6B5578]</td><td></td></tr>
    <tr><td>006B18FD</td><td>8896 78556B00</td><td>mov byte ptr ds:[esi+6B5578],dl</td><td></td></tr>
    <tr><td>006B1903</td><td>8898 7B556B00</td><td>mov byte ptr ds:[eax+6B557B],bl</td><td></td></tr>
    <tr><td>006B1909</td><td>83C0 04</td><td>add eax,4</td><td>eax:"FLAG_SRORGLnTh2Q5fTwu"</td></tr>
    <tr><td>006B190C</td><td>3D 00010000</td><td>cmp eax,100</td><td>eax:"FLAG_SRORGLnTh2Q5fTwu"</td></tr>
    <tr><td>006B1911</td><td>0F8C 09FFFFFF</td><td>jl riddle.6B1820</td><td></td></tr>
    <tr><td>006B1917</td><td>33C0</td><td>xor eax,eax</td><td>eax:"FLAG_SRORGLnTh2Q5fTwu"</td></tr>
    <tr><td>006B1919</td><td>32D2</td><td>xor dl,dl</td><td></td></tr>
    <tr><td>006B191B</td><td>32DB</td><td>xor bl,bl</td><td></td></tr>
    <tr><td>006B191D</td><td>3945 0C</td><td>cmp dword ptr ss:[ebp+C],eax</td><td></td></tr>
    <tr><td>006B1920</td><td>7E 4A</td><td>jle riddle.6B196C</td><td></td></tr>
    <tr><td>006B1922</td><td>EB 03</td><td>jmp riddle.6B1927</td><td></td></tr>
    <tr><td>006B1924</td><td>8A5D FF</td><td>mov bl,byte ptr ss:[ebp-1]</td><td></td></tr>
    <tr><td>006B1927</td><td>FEC2</td><td>inc dl</td><td></td></tr>
    <tr><td>006B1929</td><td>0FB6F2</td><td>movzx esi,dl</td><td></td></tr>
    <tr><td>006B192C</td><td>8A8E 78556B00</td><td>mov cl,byte ptr ds:[esi+6B5578]</td><td></td></tr>
    <tr><td>006B1932</td><td>02D9</td><td>add bl,cl</td><td></td></tr>
    <tr><td>006B1934</td><td>0FB6FB</td><td>movzx edi,bl</td><td></td></tr>
    <tr><td>006B1937</td><td>885D FF</td><td>mov byte ptr ss:[ebp-1],bl</td><td></td></tr>
    <tr><td>006B193A</td><td>0FB69F 78556B00</td><td>movzx ebx,byte ptr ds:[edi+6B5578]</td><td>ebx:"\8k"</td></tr>
    <tr><td>006B1941</td><td>889E 78556B00</td><td>mov byte ptr ds:[esi+6B5578],bl</td><td></td></tr>
    <tr><td>006B1947</td><td>888F 78556B00</td><td>mov byte ptr ds:[edi+6B5578],cl</td><td></td></tr>
    <tr><td>006B194D</td><td>0FB69E 78556B00</td><td>movzx ebx,byte ptr ds:[esi+6B5578]</td><td>ebx:"\8k"</td></tr>
    <tr><td>006B1954</td><td>8B75 08</td><td>mov esi,dword ptr ss:[ebp+8]</td><td>[ebp+08]:"\8k"</td></tr>
    <tr><td>006B1957</td><td>02D9</td><td>add bl,cl</td><td></td></tr>
    <tr><td>006B1959</td><td>0FB6CB</td><td>movzx ecx,bl</td><td>ecx:Ordinal#6969+73</td></tr>
    <tr><td>006B195C</td><td>0FB689 78556B00</td><td>movzx ecx,byte ptr ds:[ecx+6B5578]</td><td>ecx:Ordinal#6969+73</td></tr>
    <tr class="highlight"><td>006B1963</td><td>300C30</td><td>xor byte ptr ds:[eax+esi],cl</td><td>★ 1バイト暗号化（重要）</td></tr>
    <tr><td>006B1966</td><td>40</td><td>inc eax</td><td>eax:"FLAG_SRORGLnTh2Q5fTwu"</td></tr>
    <tr><td>006B1967</td><td>3B45 0C</td><td>cmp eax,dword ptr ss:[ebp+C]</td><td></td></tr>
    <tr><td>006B196A</td><td>7C B8</td><td>jl riddle.6B1924</td><td></td></tr>
    <tr><td>006B196C</td><td>5F</td><td>pop edi</td><td></td></tr>
    <tr><td>006B196D</td><td>5E</td><td>pop esi</td><td></td></tr>
    <tr><td>006B196E</td><td>5B</td><td>pop ebx</td><td>ebx:"\8k"</td></tr>
    <tr><td>006B196F</td><td>8BE5</td><td>mov esp,ebp</td><td></td></tr>
    <tr><td>006B1971</td><td>5D</td><td>pop ebp</td><td></td></tr>
    <tr><td>006B1972</td><td>C3</td><td>ret</td><td></td></tr>
  </tbody>
</table>

</body>
</html>


アドレス006B1A02のcmp edx,dword ptr ds:[ecx]でレジスタの値を確認
- ECX:006B51F4のダンプを確認 … 正解暗号文
- ESX:0117ED18のダンプを確認 … 暗号化済み入力

|レジスタ|値|
|:---|:---|
|EAX | 00000015|
|EBX | 0117F568     "\\8k"|
|ECX | 006B51F4     riddle.006B51F4|
|EDX | 0D1BFFBF|
|EBP | 0117ED3C|
|ESP | 0117ED0C|
|ESI | 0117ED18|
|EDI | 00000111     L'đ'|
|EIP | 006B1A02     riddle.006B1A02|
|EFLAGS | 00000304     L'̄'|
|ZF | 0|
|OF | 0|
|CF | 0     L'Ā'|
|PF | 1|
|SF | 0|
|TF | 1     L'ā'|
|AF | 0|
|DF | 0|
|IF | 1|
|LastError | 00000000 (ERROR_SUCCESS)|
|LastStatus | C0000034 (STATUS_OBJECT_NAME_NOT_FOUND)|
|GS | 002B|
|ES | 002B|
|CS | 0023|
|FS | 0053|
|DS | 002B|
|SS | 002B|
|ST(0) | 400B8C28000000000000|
|ST(1) | 400AAE01D3B597796000|
|ST(2) | 40079AAC4A6886A4C800|
|ST(3) | 400BE1C8000000000000|
|ST(4) | 3FFF8000000000000000|
|ST(5) | 3FFF8000000000000000|
|ST(6) | 40038000000000000000|
|ST(7) | 40038000000000000000|
|x87TagWord | FFFF|
|x87ControlWord | 027F|
|x87StatusWord | 4020|
|x87TW_0 | 3 (空)|
|x87TW_1 | 3 (空)|
|x87TW_2 | 3 (空)|
|x87TW_3 | 3 (空)|
|x87TW_4 | 3 (空)|
|x87TW_5 | 3 (空)|
|x87TW_6 | 3 (空)|
|x87TW_7 | 3 (空)|
|x87SW_B | 0     L'Ā'|
|x87SW_C3 | 1|
|x87SW_TOP | 0 (ST0=x87r0)|
|x87SW_C2 | 0|
|x87SW_C1 | 0|
|x87SW_O | 0|
|x87SW_ES | 0|
|x87SW_SF | 0     L'Ā'|
|x87SW_P | 1|
|x87SW_U | 0|
|x87SW_Z | 0|
|x87SW_D | 0|
|x87SW_I | 0|
|x87SW_C0 | 0|
|x87CW_IC | 0|
|x87CW_RC | 0 (最近接丸め)|
|x87CW_PC | 2 (Real8)|
|x87CW_PM | 1|
|x87CW_UM | 1|
|x87CW_OM | 1|
|x87CW_ZM | 1|
|x87CW_DM | 1     L'ā'|
|x87CW_IM | 1|
|MxCsr | 00001FA0|
|MxCsr_FZ | 0|
|MxCsr_PM | 1|
|MxCsr_UM | 1|
|MxCsr_OM | 1|
|MxCsr_ZM | 1|
|MxCsr_IM | 1|
|MxCsr_DM | 1|
|MxCsr_DAZ | 0     L'Ā'|
|MxCsr_PE | 1|
|MxCsr_UE | 0|
|MxCsr_OE | 0|
|MxCsr_ZE | 0|
|MxCsr_DE | 0|
|MxCsr_IE | 0|
|MxCsr_RC | 0 (最近接丸め)|
|XMM0  | 00000000000000000000000000000000|
|XMM1  | 00000000000000000000000000000000|
|XMM2  | 00000000000000000000000000000000|
|XMM3  | 00000000000000000000000000000000|
|XMM4  | 00000000000000000000000000000000|
|XMM5  | 00000000000000000000000000000000|
|XMM6  | 00000000000000000000000000000000|
|XMM7  | 00010000000100000000000000000000|
|K0  | 0000000000000000|
|K1  | 0000000000000000|
|K2  | 0000000000000000|
|K3  | 0000000000000000|
|K4  | 0000000000000000|
|K5  | 0000000000000000|
|K6  | 0000000000000000|
|K7  | 0000000000000000|
|DR0 | 00000000|
|DR1 | 00000000|
|DR2 | 00000000|
|DR3 | 00000000|
|DR6 | 00000000|
|DR7 | 00000000|

|アドレス|バイト列|ASCII|
|---|:---|:---|
|006B51F4 | BF FF 1B 0D 47 A7 18 4F CB D6 5C 59 95 61 73 57 | ¿ÿ..G§.OËÖ\Y.asW  |
|006B5204 | B3 D1 94 9F AC 00 00 00 00 00 00 00 00 00 00 00 | ³Ñ..¬...........  |

|アドレス|バイト列|ASCII|
|---|:---|:---|
|0117ED18 | BF FF 1B 0D 47 B9 0F 79 CB A5 4A 51 F5 48 75 4E | ¿ÿ..G¹.yË¥JQõHuN  |
|0117ED28 | C0 E0 F7 DB B2 00 17 01 C3 EF 17 01 C8 F1 17 01 | Àà÷Û²...Ãï..Èñ..  |
